<a href="https://colab.research.google.com/github/donggeunlee77/MONAI-tutorial-exercise/blob/main/MONAI-2D-Segmentation-tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# unet_training_dict.py 파일로 연습

In [ ]:
import logging ## 로그 기록 및 출력하는 것. 학습 진행 상황이나 오류 출력용
import os ## 운영체제와 파일 시스템 조작.
import sys ## 파이썬 실행환경 제어
import tempfile ## 임시파일 및 폴더 생성
from glob import glob ## 특정 패턴의 파일 검색 - 예를들면 glob("경로/*.png") 이렇게 하면 해당 경로 폴더 안에 있는 모든 PNG 파일의 경로를 리스트로 가져와줌.

import torch ## 파이토치 핵심라이브러리
from PIL import Image ## 이미지파일 읽고 처리하는 라이브러리 (Python Imaging Library)
from torch.utils.tensorboard import SummaryWriter

import monai
from monai.data import create_test_image_2d, list_data_collate, decollate_batch, DataLoader ## 순서대로 테스트용 2D 이미지와 정답마크 생성 / 여러 데이터 샘플을 하나의 Batch로 결합 / Batch를 개별샘플로 분리 / 데이터를 Batch 단위로 모델에 공급
from monai.inferers import sliding_window_inference ## 슬라이딩 윈도우 추론 : 큰 이미지를 한번에 모델에 입력하는 대신 작은 영역으로 나누어 순차적으로 예측한 다음 결과를 합치는 방식. GPU 메모리 문제로.
from monai.metrics import DiceMetric ## Segmentation 성능을 평가하는 Dice score : 예측된 영역과 실제 정답 영역이 얼마나 겹치는지
from monai.transforms import (
    Activations,
    EnsureChannelFirstd,
    AsDiscrete, ## 연속적인 출력을 이산적인 label로 변환
    Compose,
    LoadImaged,
    RandCropByPosNegLabeld, ##정답 영역의 양성, 음성 비율을 고려하여 이미지 crop하는 것. 전체 영상에서 병변이 차지하는 비율이 매우 적을 수 있는데, 무작위로 crop하면 병변이 없는 영역만 계속 학습할수도 있으니.
    RandRotate90d,
    ScaleIntensityd ##이미지 밝기 및 intensity 정규화
) ## d 는 dictionary 기반 transform을 의미함. 파이썬 딕셔너리(Dictionary)는 키(Key)와 값(Value)의 쌍을 한 쌍으로 갖는 자료형. 예를들어 {"img" : "image001.png", "seg" : "label001.png"}, user = {"name": "홍길동", "age": 25, "email": "hong@example.com"}
from monai.visualize import plot_2d_or_3d_image ## 2D또는 3D이미지를 시각화하는 기능. segmentation에서는 모델이 실제로 병변을 잘 찾아내고 있는지 눈으로 확인하는 것이 중요하기 떄문에 자주 활용됨.


이후, 전체 구조는 아래와 같은 구조로 이루어짐
1) 프로그램 실행 (if __name__ == "__main__": )
2) 임시 데이터 저장 공간 생성 (tempfile.TemporaryDictionary())
3) main(tempdir) 실행

main() 내부에서는
(1) 가상 영상 및 정답 mask 생성
(2) train/validation 데이터 분리
(3) 이미지 전처리(transform)
(4) Dataset 및 DataLoader 생성
(5) U-Net 모델 및 Loss, Optimizer 설정
(6) 모델 학습(training)
(7) validation 및 dice 평가
(8) 최적 모델 저장 및 Tensorboard 기록
의 과정이 일어남. 즉 main(0은 이 모든 과정을 순서대로 실행하도록 묶어놓은 함수.

main은 특별한 예약어가 아니라 개발자가 임의로 붙인 함수 이름. 예를 들어
def main():
 print("Hello")
 print("MONAI")

main()
을 실행하면
Hello
MONAI
이렇게 출력됨.

근데 우리 코드는
def main(tempdir): 이렇게 괄호 안에 변수가 들어있음. 즉 이는 함수를 실행할 때 외부에서 데이터를 전달받겠다는 의미.
글고 if __name__ == "__main__"는 이 파일을 직접 실행했을 때만 아래 코드를 실행하는 것이라고 하네

In [ ]:
def main(tempdir):
  monai.config.print_config()  # 현재 설치된 MONAI 및 관련 라이브러리의 버전과 환경 출력
  logging.basicConfig(stream=sys.stdout, level=logging.INFO) ## 프로그램 실행 중 발생하는 로그를 설정 // 이까지는 학습에 들어가기 전 실행 환경을 확인하는 과정.

  ## create a temporary dictionary and 40 random image, mask pairs
  print(f"generating synthetic data to {tempdir} (this may take a while)")
  for i in range(40):
    im, seg = create_test_image_2d(128, 128, num_seg_classes=1) ## 원본이미지 im과 정답 segmentation mask인 seg 가 생성됨. for 구문에 따라 총 40개겠지?
    Image.fromarray((im * 255).astype("uint8")).save(os.path.join(tempdir, f"img{i:d}.png")) ## im 혹은 seg * 255는 0~1범위의 픽셀값을 0~255로 변환, astype("uint8")은 8bit 정수형으로 변환, Image.fromarray는 배열을 이미지 객체로 변환.
    Image.fromarray((seg * 255).astype("uint8")).save(os.path.join(tempdir, f"seg{i:d}.png")) ## 결과적으로 임시폴더에 img0.png ~ img39.png / seg0.png~seg39.png 가 생성됨.

  images = sorted(glob(os.path.join(tempdir, "img*.png"))) ## glob은 특정 패턴에 해당하는 파일 검색, sorted는 파일 정렬
  segs = sorted(glob(os.path.join(tempdir, "seg*.png")))
  train_files = [{"img" : img, "seg" : seg} for img, seg in zip(images[:20], segs[:20])] ## zip은 iterable한 객체 여러개를 받아 동일한 인덱스의 요소를 튜플로 묶어주는 반복자(iterator)를 반환해줌. 예를들어 numbers = [1,2,3], letters=['A','B','C'] 이면 zip(numbers, letters)는 [(1, 'A'), (2, 'B'), (3, 'C')] 이렇게 나오는것
  val_files = [{"img" : img, "seg" : seg} for img, seg in zip(images[-20:], segs[-20:])] ## train_files는 앞에서 20개, val_files는 뒤에서 20개

  ## define transforms for image and segmentation
  train_transforms = Compose(
      [
          LoadImaged(keys=["img", "seg"]),
          EnsureChannelFirstd(keys=["img", "seg"]),
          ScaleIntensityd(keys=["img", "seg"]), ## 픽셀 intensity의 정규화
          RandCropByPosNegLabeld(  ## 정답 mask 를 기준으로 무작위 crop
              keys=["img", "seg"], label_key="seg", spatial_size=[96,96], pos=1, neg=1, num_samples=4
          ), ## 원본영상은 128*128이지만, 학습시에는 96*96 크기로 잘라서 사용하겠다. 그리고 정답 mask를 기준으로 양성 영역과 음성 영역을 고려해 crop 하겠다. pos=1과 neg=1은 각각 영역을 포함하는 crop의 선택 가중치, num_samples=4는 이미지 하나당 4개의 crop 생성.
          RandRotate90d(keys=["img", "seg"], prob=0.5, spatial_axes=[0,1]) ## 핵심은, 원본 영상과 정답 mask를 동시에 회전시킴.
      ]
  )
  val_transforms = Compose(
      [
          LoadImaged(keys=["img", "seg"]),
          EnsureChannelFirstd(keys=["img", "seg"]),
          ScaleIntensityd(keys=["img", "seg"])
      ]
  ) # validation은 crop이나 회전 안하고 일관된 조건에서 수행.

  ## define dataset and data loader